# Lecture 23 — Introduction to Raster Data with Rasterio

In the previous lectures, we worked mainly with **vector data** such as points, lines, and polygons. We now begin working with **raster data**.

A raster represents geographic space as a regular grid of cells or pixels. Each cell stores a value. Depending on the dataset, that value may represent elevation, temperature, rainfall, land-cover class, reflectance, vegetation index, or another gridded variable.

For this lecture, we use a **Digital Elevation Model (DEM)** stored as a GeoTIFF:

`C:\Teaching\IIT Tirupati\UISA\2026\uisa_codes\data\rasters_COP30\output_hh.tif`

We will learn how to open the raster, inspect its properties, understand rows, columns, bands, resolution, extent, CRS and NoData, read raster values into Python, briefly introduce NumPy arrays, and plot the DEM.


## 1. Rasterio

**Rasterio** is a Python library used for working with geospatial raster data.

A useful comparison is:

- **GeoPandas** → mainly used for vector data
- **Rasterio** → mainly used for raster data

Rasterio gives us access to both the **cell values** stored in a raster and the **spatial information** that tells us where those cells are located on Earth.

Common raster datasets include GeoTIFFs of satellite imagery, Digital Elevation Models, land-cover rasters, rainfall rasters, and temperature rasters.


In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from rasterio.plot import show

## 2. Define the file path

The raster is stored in a separate folder. Students should change this path according to the location of the data on their own computer.


In [ ]:
data_path = r"C:\Teaching\IIT Tirupati\UISA\2026\uisa_codes\data\rasters_COP30"

raster_path = data_path + r"\output_hh.tif"

raster_path

## 3. Open the raster

We use `rasterio.open()` to open the GeoTIFF.

This is similar in spirit to using `gpd.read_file()` for vector data, although Rasterio returns a raster dataset object rather than a GeoDataFrame.

At this stage, the raster is open, but we have not yet read all of its pixel values into a NumPy array.


In [ ]:
src = rasterio.open(raster_path)

src

## 4. Inspect the basic raster properties

A raster is more than an image. To interpret it correctly, we should know its:

- CRS
- number of rows and columns
- number of bands
- resolution or cell size
- spatial extent
- data type
- NoData value
- transform


In [ ]:
print("CRS:", src.crs)
print("Width / columns:", src.width)
print("Height / rows:", src.height)
print("Number of bands:", src.count)
print("Resolution:", src.res)
print("Bounds:", src.bounds)
print("Data type:", src.dtypes)
print("NoData value:", src.nodata)

### Rows and columns

A raster is organised as a rectangular grid.

- `width` = number of **columns**
- `height` = number of **rows**

The number of cells in one band is:

`rows × columns`


In [ ]:
number_of_cells = src.width * src.height

print("Number of cells in one band:", number_of_cells)

### Resolution

`src.res` gives the cell size in the units of the raster CRS.

For example, if the resolution is approximately `(30, 30)`, each pixel represents approximately **30 m × 30 m** on the ground, provided the CRS uses metres.

A smaller cell size generally means finer spatial detail.


In [ ]:
src.res

**Note:** Since this raster uses **EPSG:4326**, the resolution is expressed in **degrees**, not metres. A resolution of `0.00027778°` is approximately **1 arc-second**, corresponding to roughly **30 m at the equator**. The actual ground distance, especially in the east–west direction, varies with latitude.

### Spatial extent or bounds

The raster bounds describe the total geographic area covered by the raster.

Rasterio reports:

- `left`
- `bottom`
- `right`
- `top`


In [ ]:
print("Left:", src.bounds.left)
print("Bottom:", src.bounds.bottom)
print("Right:", src.bounds.right)
print("Top:", src.bounds.top)

### Coordinate Reference System (CRS)

Just like vector data, raster data have a Coordinate Reference System.

The CRS tells us how raster coordinates relate to locations on Earth. This becomes important when we later combine raster and vector data.


In [ ]:
src.crs

## 5. Bands

A raster may contain one band or several bands.

### Single-band raster

A single-band raster stores **one value for each cell location**. A DEM is usually a single-band raster: each pixel stores an elevation value.

### Multi-band raster

A multi-band raster stores several layers of values for the same geographic locations. Satellite imagery may contain separate bands for red, green, blue, near-infrared, and other wavelengths.

Rasterio numbers bands starting from **1**, not 0.

Therefore, `src.read(1)` means **read Band 1**.


In [ ]:
print("Number of bands:", src.count)
print("Band indexes:", src.indexes)
print("Band descriptions:", src.descriptions)
print("Colour interpretation:", src.colorinterp)

## 6. Reading the raster values

Until now, we have mostly inspected metadata.

To work with the actual cell values, we use `src.read()`.

If we read one band using `src.read(1)`, Rasterio returns the values as a **NumPy array**.


In [ ]:
band1 = src.read(1)

type(band1)

## A brief introduction to NumPy arrays

**NumPy** is a Python library for working efficiently with numerical data.

A two-dimensional NumPy array closely matches the structure of a raster grid:

```text
12   15   18   20
14   17   19   22
16   20   24   27
18   22   26   30
```

For a raster band:

- array rows correspond to raster rows;
- array columns correspond to raster columns;
- each array element corresponds to one raster cell.

This is why NumPy is so useful for raster analysis.

A simple way to remember the relationship is:

> **Rasterio handles the geospatial raster; NumPy helps us work with the cell values.**


In [ ]:
print("Array shape:", band1.shape)
print("Number of dimensions:", band1.ndim)
print("Data type:", band1.dtype)

### Understanding `shape`

For a single raster band, the array shape is:

`(rows, columns)`

This corresponds directly to:

- `src.height` → rows
- `src.width` → columns


In [ ]:
print("Raster height:", src.height)
print("Raster width:", src.width)
print("Band 1 shape:", band1.shape)

### Looking at actual cell values

Because the raster band is now a NumPy array, we can inspect individual cells or small sections of the grid.

`band1[0, 0]` means the value in the **first row and first column**.

NumPy indexing starts from **0**.


In [ ]:
print("First cell value:", band1[0, 0])

print("\nFirst 5 rows and first 5 columns:")
display(band1[:5, :5])

## 7. NoData values

Not every cell in a raster must contain a valid observation.

Some cells may represent areas outside the study region, outside the source data coverage, or places where information is missing.

These cells may be assigned a special **NoData** value.

The NoData value should not be treated as a real elevation, temperature, rainfall value, etc.


In [ ]:
print("NoData value:", src.nodata)

Rasterio can read a raster as a **masked array**.

Using `masked=True` tells Rasterio to mask cells identified as NoData. This is useful when calculating statistics because missing cells should not influence the result.


In [ ]:
band1_masked = src.read(1, masked=True)

band1_masked

## 8. Basic raster statistics

Once the raster values are available as an array, we can calculate simple statistics.

For a DEM, these statistics describe elevation.

We use the masked array so that NoData cells are excluded.


In [ ]:
print("Minimum elevation:", band1_masked.min())
print("Maximum elevation:", band1_masked.max())
print("Mean elevation:", band1_masked.mean())

## 9. The raster transform

The NumPy array itself only knows rows and columns.

It does not automatically know where those rows and columns are located geographically.

The raster **transform** connects array positions to geographic coordinates.

A useful way to remember this is:

> **Array = values**

> **Transform = where those values are located**

We do not need to study the mathematics of the affine transformation at this stage.


In [ ]:
src.transform

## 10. Raster metadata / profile

Rasterio stores the important characteristics of the raster together in a **profile**.

The profile includes information such as:

- file driver;
- width and height;
- number of bands;
- data type;
- CRS;
- transform; and
- NoData value.

This becomes especially useful later when we create and save new rasters.


In [ ]:
src.profile

## 11. Plotting the DEM with Rasterio

Rasterio provides a convenient plotting function called `show()`.

Because Rasterio knows the raster transform, it can plot the raster using geographic coordinates rather than simply treating it as an ordinary image.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

show(
    band1_masked,
    transform=src.transform,
    ax=ax,
    cmap="terrain"
)

ax.set_title("Digital Elevation Model")

plt.show()

### Why use a colour map?

The raster contains numerical elevation values.

A colour map converts different numerical values into different colours so that spatial variation is easier to interpret visually.

Here we use `cmap="terrain"` because it is intuitive for elevation.

The colour is only a visual representation. The actual information remains the numerical elevation value stored in each pixel.


## 12. Plotting with a colour bar

We can also use Matplotlib's `imshow()` to plot the NumPy array directly.

This gives us more control over elements such as a colour bar.

Because `imshow()` mainly sees an array, we provide the raster extent so that the axes correspond to geographic coordinates.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

image = ax.imshow(
    band1_masked,
    cmap="terrain"
)

plt.colorbar(
    image,
    ax=ax,
    label="Elevation (m)"
)

ax.set_title("Digital Elevation Model")

plt.show()

## 13. Reading all bands at once

There is an important difference between:

`src.read(1)`

and:

`src.read()`

`src.read(1)` reads **one band** and normally returns a two-dimensional array:

`(rows, columns)`

`src.read()` reads **all bands** and returns a three-dimensional array:

`(bands, rows, columns)`

For a single-band DEM, the first dimension contains only one band.


In [ ]:
all_bands = src.read()

print("Shape when all bands are read:", all_bands.shape)
print("Number of dimensions:", all_bands.ndim)

For a one-band DEM, the output may look like:

`(1, rows, columns)`

The first `1` represents the number of bands.

This becomes much more important when we later work with multi-band satellite imagery.


## 14. Rasterio dataset versus NumPy array

It is useful to distinguish these two objects.

### Rasterio dataset: `src`

Contains spatial information and metadata such as:

- CRS;
- bounds;
- resolution;
- transform;
- number of bands;
- width and height.

### NumPy array: `band1`

Contains the actual numerical cell values from Band 1.

Conceptually:

```text
GeoTIFF
   ↓
Rasterio opens the raster
   ↓
Spatial information + pixel values
   ↓
Pixel values can be read into a NumPy array
```


## 15. Summary

The main ideas from this lecture are:

- Raster data represent space as a **grid of cells**.
- Each cell stores a value.
- `width` gives the number of columns.
- `height` gives the number of rows.
- `res` gives the cell size or spatial resolution.
- `bounds` gives the spatial extent.
- `crs` tells us how raster coordinates relate to locations on Earth.
- A raster may contain one or several **bands**.
- A DEM is generally a **single-band raster** containing elevation values.
- `src.read(1)` reads Band 1.
- Raster values are read into a **NumPy array**.
- NoData cells should not be treated as real observations.
- The transform connects array positions to geographic coordinates.
- Rasterio can plot a georeferenced raster and later be used for clipping, masking, reprojection, and writing new rasters.


# Class Exercise 19 — Explore a DEM

Plot the elevation map of your hometown/nearest city

1. Open the DEM using Rasterio.
2. Report its CRS.
3. Report the number of rows and columns.
4. Report the number of bands.
5. Report the raster resolution.
6. Report the raster bounds.
7. Report the NoData value.
8. Read Band 1 as a masked array.
9. Find the minimum, maximum, and mean elevation.
10. Plot the DEM with a suitable colour map and colour bar.

In one or two sentences, describe what the DEM tells us about the variation in elevation across the study area.
